# Exercise:  Cu equation of state

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ase.build import bulk
from ase.eos import EquationOfState
from ase.units import kJ

from mace.calculators import MACECalculator

# https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
#  MODEL is MACE-MP-0 medium
MODEL = './2023-12-03-mace-128-L1_epoch-199.model'
calculator = MACECalculator(model_paths=MODEL, 
                            device='cpu',   # 'cuda' for GPU
                            default_dtype='float64')


# Initial crystal structure
atoms = bulk('Cu', 'fcc', a=3.615)
atoms.calc = calculator 

# Save the original cell
cell0 = atoms.cell.copy()

volumes = []
energies = []

# Uniformly scale the lattice vectors
# scale=1 corresponds to the initial lattice parameter
for scale in np.linspace(0.94, 1.06,  nine := 9):

    atoms.set_cell(cell0 * scale, scale_atoms=True)

    V = atoms.get_volume()
    E = atoms.get_potential_energy()

    volumes.append(V)
    energies.append(E)

    print(f"scale = {scale:.4f}, V = {V:.4f} Å^3, E = {E:.6f} eV")

# Fit E(V)
eos = EquationOfState(volumes, energies, eos='birchmurnaghan')

V0, E0, B0 = eos.fit()

# ASE returns bulk modulus in eV / Å^3
B0_GPa = B0 / kJ * 1.0e24

print()
print(f"Equilibrium volume = {V0:.6f} Å^3")
print(f"Minimum energy     = {E0:.6f} eV")
print(f"Bulk modulus       = {B0_GPa:.3f} GPa")
print(f" Experimental Bulk modulus = 140 GPa (https://en.wikipedia.org/wiki/Copper)")

# Plot EOS
# eos.plot(filename='Cu_EOS.png')
plt.show()